# MEN 4338 · Week 8 — Ordinary kriging at a point

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W8_ordinary_kriging_point.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Kriging is a weighted average like IDW, but the weights come from the **variogram**. They are the weights that
make the estimate **unbiased** and the estimation **error variance as small as possible**. In exchange we get
something IDW cannot give: the **kriging variance**, a measure of how reliable each estimate is.

**What you will practise:** the ordinary kriging system · solving it with NumPy · the kriging variance ·
the screening effect · exactness · the effect of the nugget.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w8/q1.py', 'tests/q1.py'), ('tests/w8/q2.py', 'tests/q2.py'), ('tests/w8/q3.py', 'tests/q3.py'), ('tests/w8/q4.py', 'tests/q4.py'), ('tests/w8/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def gamma_bassam(dx, dy):
    """Bassam model (week 5): nugget 0.06 + spherical 0.20, ranges 300 m (N045) and 150 m (N135)."""
    return spherical(aniso_h(dx, dy, 45, 300, 150), 0.06, 0.20, 300)

samples = pd.read_csv("bassam_samples.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()

## 1. The ordinary kriging system

We look for $z^* = \sum_i \lambda_i z_i$ with $\sum_i \lambda_i = 1$ (no bias when the mean is unknown)
that minimises the error variance. With a Lagrange multiplier $\mu$ for the constraint, the solution is
the linear system

$$
\begin{bmatrix} \gamma_{11} & \cdots & \gamma_{1n} & 1 \\ \vdots & & \vdots & \vdots \\
\gamma_{n1} & \cdots & \gamma_{nn} & 1 \\ 1 & \cdots & 1 & 0 \end{bmatrix}
\begin{bmatrix} \lambda_1 \\ \vdots \\ \lambda_n \\ \mu \end{bmatrix} =
\begin{bmatrix} \gamma_{10} \\ \vdots \\ \gamma_{n0} \\ 1 \end{bmatrix}
$$

where $\gamma_{ij} = \gamma(\mathbf{x}_i - \mathbf{x}_j)$ and $\gamma_{i0} = \gamma(\mathbf{x}_i - \mathbf{x}_0)$.
The **kriging variance** is

$$\sigma^2_{OK} = \sum_i \lambda_i \gamma_{i0} + \mu$$

It depends on the geometry of the data and on the variogram, **not on the grade values**.

## 2. What the weights do

* Samples **closer** to the target, measured in the anisotropic distance, get more weight.
* Samples **hidden behind** another sample get little weight (screening effect).
* **Clustered** samples share the weight of one sample: kriging declusters by itself.
* Weights can be slightly **negative** when there is strong screening.

In [ ]:
xs = np.array([100., 150, 120, 200]); ys = np.array([100., 120, 170, 160])
vs = np.array([1.2, 0.8, 1.5, 2.1]); x0, y0 = 140., 140.
plt.figure(figsize=(4, 4))
plt.scatter(xs, ys, s=60); plt.scatter([x0], [y0], marker="x", s=80, c="red")
for k, (a, b) in enumerate(zip(xs, ys)): plt.annotate(f"{k}: {vs[k]}", (a + 3, b + 3))
plt.gca().set_aspect("equal"); plt.title("4 samples and the target"); plt.show()

---
## Exercises

Q1 to Q3 use the 4 samples above (`xs`, `ys`, `vs`) and the target `(x0, y0)`.

### Q1. Build the system

Write `ok_matrices(xs, ys, x0, y0, gamma)` returning `(A, b)` as defined above. `gamma(dx, dy)` is a variogram
model of the separation vector, such as `gamma_bassam`.

*Hint:* `xs[:, None] - xs[None, :]` gives the matrix of all differences.

In [ ]:
def ok_matrices(xs, ys, x0, y0, gamma):
    ...

A, b = ok_matrices(xs, ys, x0, y0, gamma_bassam)
np.round(A, 3), np.round(b, 3)

In [ ]:
grader.check("q1")

### Q2. Solve it

Solve the system: store the 4 weights in `weights` and the Lagrange multiplier in `mu`. Which sample gets the
largest weight? Is it the closest one in plain distance?

In [ ]:
sol = ...
weights = ...
mu = ...
np.round(weights, 3), mu

In [ ]:
grader.check("q2")

### Q3. Estimate and kriging variance

Store the estimate in `ok_est` and the kriging variance in `ok_var`.

In [ ]:
ok_est = ...
ok_var = ...
ok_est, ok_var

In [ ]:
grader.check("q3")

### Q4. Nugget, screening and exactness

1. With a **pure nugget** model of sill 1, `gamma_nugget = lambda dx, dy: np.where(np.hypot(dx, dy) > 0, 1.0, 0.0)`,
   compute the weights of the 4 samples (`weights_nugget`) and the kriging variance (`ok_var_nugget`).
   What does kriging become?
2. With `gamma_bassam`, compute the weights `weights_screen` for three samples at (140, 140), (160, 160) and
   (165, 165), and the target (140, 140). What happens to the first weight, and why?

In [ ]:
gamma_nugget = lambda dx, dy: np.where(np.hypot(dx, dy) > 0, 1.0, 0.0)
weights_nugget = ...
ok_var_nugget = ...

weights_screen = ...
weights_nugget, ok_var_nugget, np.round(weights_screen, 4)

In [ ]:
grader.check("q4")

### Q5. Kriging the Bassam data

Write `ok_point(x0, y0, x, y, v, gamma, nmax=16)` that krigs a point from its `nmax` closest samples and
returns `(estimate, variance)`. Store in `ok_targets` the results at (312, 457), (641, 213) and (906, 782),
and in `var_far` the kriging variance at (1190, 990).

In [ ]:
def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    ...

ok_targets = ...
var_far = ...
ok_targets, var_far

In [ ]:
grader.check("q5")

### Going further (not checked)

* Move the target along a line from (0, 500) to (1200, 500) and plot the kriging variance. Where is it lowest?
* Replace `gamma_bassam` by an isotropic model (range 225 m) at (641, 213). How do the weights change?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()